In [1]:
# --- config ---
ROW_COUNT = 50_000_000   # only this is configurable

In [2]:
import duckdb, json, os

SEED = 0.42                      # hardcoded for reproducibility (data + fail-row volumes)
OUT_DIR = os.getcwd()            # output next to this notebook
PARQUET_PATH = os.path.join(OUT_DIR, 'dataset.parquet')
RULES_PATH   = os.path.join(OUT_DIR, 'rules.json')

con = duckdb.connect()
con.execute(f'SELECT setseed({SEED})')

# Half the rules FAIL, half PASS. Failing row-rules each carry a random-but-seeded
# volume of bad rows (~0.5-2.5%); pass rules are perfectly clean; failing aggregate
# rules fail as a whole-dataset check.
GEN_SQL = "CREATE OR REPLACE TABLE dataset AS\nSELECT\n  CASE WHEN random() < 0.05 THEN NULL ELSE 'v' || (i % 1000) END AS c_any_nullable,\n  CASE WHEN (i % 100000) < 2099 THEN NULL ELSE 'id' || i END AS c_any_required,\n  CASE WHEN i = 1 THEN 999999 ELSE round(random()*1000, 2) END AS n_max,\n  round(random()*1000, 2) AS n_min,\n  CASE WHEN i = 2 THEN 1e14 ELSE round(random()*1000, 2) END AS n_sum,\n  round(random()*1000, 2) AS n_median,\n  round(random()*1000, 2) AS n_stddev,\n  list_element(['A','A','A','B','C'], 1 + (floor(random()*5))::INT) AS cat_mode,\n  CASE WHEN (i % 100000) < 563 THEN 500 ELSE round(random()*100, 2) END AS n_between,\n  CASE WHEN (i % 100000) < 1188 THEN repeat('x',20) ELSE repeat('x', 3 + (floor(random()*8))::INT) END AS s_len,\n  CASE WHEN (i % 100000) < 1058 THEN 'Q' ELSE list_element(['X','Y','Z'], 1 + (floor(random()*3))::INT) END AS cat_in,\n  list_element(['OK','FINE','GOOD'], 1 + (floor(random()*3))::INT) AS cat_not_in,\n  CASE WHEN (i % 100000) < 2341 THEN 'bad' ELSE 'ABC' || lpad(((i % 1000))::VARCHAR, 3, '0') END AS s_regex,\n  list_element(['alpha','bravo','charlie'], 1 + (floor(random()*3))::INT) AS s_regex_not,\n  i AS id_prop,\n  CASE WHEN i < 100 THEN 999 ELSE (i % 12) END AS cat_distinct,\n  CASE WHEN (i % 100000) < 2192 THEN 0 ELSE round(random()*1000 + 1000, 2) END AS n_gt_hi,\n  round(random()*1000, 2) AS n_gt_lo,\n  (DATE '2024-01-01' + (i % 365)::INT) AS d_eq_a,\n  (DATE '2024-01-01' + (i % 365)::INT) AS d_eq_b,\n  i AS dup_a,\n  (i * 2) AS dup_b\nFROM range({n}) t(i);"
con.execute(GEN_SQL.format(n=ROW_COUNT))
con.execute(f"COPY dataset TO '{PARQUET_PATH}' (FORMAT PARQUET)")

RULES = [
  {
    "id": "COMP_IS_NULL",
    "dimension": "Completeness",
    "business_label": "Nullable field contains nulls as expected",
    "columns": [
      "c_any_nullable"
    ],
    "result_kind": "aggregate",
    "sql": "count(*) FILTER (WHERE c_any_nullable IS NULL) >= 1",
    "expected": "PASS"
  },
  {
    "id": "COMP_NOT_NULL",
    "dimension": "Completeness",
    "business_label": "Field is always present",
    "columns": [
      "c_any_required"
    ],
    "result_kind": "row",
    "sql": "c_any_required IS NOT NULL",
    "expected": "FAIL"
  },
  {
    "id": "STAT_MAX_BETWEEN",
    "dimension": "Validity",
    "business_label": "Maximum is within range",
    "columns": [
      "n_max"
    ],
    "result_kind": "aggregate",
    "sql": "max(n_max) BETWEEN 0 AND 1000",
    "expected": "FAIL"
  },
  {
    "id": "STAT_MIN_BETWEEN",
    "dimension": "Validity",
    "business_label": "Minimum is within range",
    "columns": [
      "n_min"
    ],
    "result_kind": "aggregate",
    "sql": "min(n_min) BETWEEN 0 AND 1000",
    "expected": "PASS"
  },
  {
    "id": "STAT_SUM_BETWEEN",
    "dimension": "Validity",
    "business_label": "Sum is within range",
    "columns": [
      "n_sum"
    ],
    "result_kind": "aggregate",
    "sql": "sum(n_sum) BETWEEN 0 AND 100000000000",
    "expected": "FAIL"
  },
  {
    "id": "STAT_MEAN_BETWEEN",
    "dimension": "Validity",
    "business_label": "Mean is within range",
    "columns": [
      "n_sum"
    ],
    "result_kind": "aggregate",
    "sql": "avg(n_sum) BETWEEN 0 AND 1000",
    "expected": "FAIL"
  },
  {
    "id": "STAT_MEDIAN_BETWEEN",
    "dimension": "Validity",
    "business_label": "Median is within range",
    "columns": [
      "n_median"
    ],
    "result_kind": "aggregate",
    "sql": "median(n_median) BETWEEN 0 AND 1000",
    "expected": "PASS"
  },
  {
    "id": "STAT_STDDEV_BETWEEN",
    "dimension": "Validity",
    "business_label": "Standard deviation is within range",
    "columns": [
      "n_stddev"
    ],
    "result_kind": "aggregate",
    "sql": "stddev_samp(n_stddev) BETWEEN 0 AND 1000",
    "expected": "PASS"
  },
  {
    "id": "STAT_MODE_IN_SET",
    "dimension": "Validity",
    "business_label": "Most common value is expected",
    "columns": [
      "cat_mode"
    ],
    "result_kind": "aggregate",
    "sql": "mode(cat_mode) IN ('A','B','C')",
    "expected": "PASS"
  },
  {
    "id": "VAL_BETWEEN",
    "dimension": "Validity",
    "business_label": "Value is within range",
    "columns": [
      "n_between"
    ],
    "result_kind": "row",
    "sql": "n_between BETWEEN 0 AND 100",
    "expected": "FAIL"
  },
  {
    "id": "VAL_LENGTH_BETWEEN",
    "dimension": "Validity",
    "business_label": "Text length is within range",
    "columns": [
      "s_len"
    ],
    "result_kind": "row",
    "sql": "length(s_len) BETWEEN 3 AND 10",
    "expected": "FAIL"
  },
  {
    "id": "VAL_IN_SET",
    "dimension": "Validity",
    "business_label": "Value is in the allowed set",
    "columns": [
      "cat_in"
    ],
    "result_kind": "row",
    "sql": "cat_in IN ('X','Y','Z')",
    "expected": "FAIL"
  },
  {
    "id": "VAL_NOT_IN_SET",
    "dimension": "Validity",
    "business_label": "Value is not in the blocked set",
    "columns": [
      "cat_not_in"
    ],
    "result_kind": "row",
    "sql": "cat_not_in NOT IN ('BAD','BANNED')",
    "expected": "PASS"
  },
  {
    "id": "VAL_REGEX_MATCH",
    "dimension": "Validity",
    "business_label": "Text matches the required pattern",
    "columns": [
      "s_regex"
    ],
    "result_kind": "row",
    "sql": "regexp_matches(s_regex,'^[A-Z]{3}[0-9]{3}$')",
    "expected": "FAIL"
  },
  {
    "id": "VAL_REGEX_NOT_MATCH",
    "dimension": "Validity",
    "business_label": "Text does not match the blocked pattern",
    "columns": [
      "s_regex_not"
    ],
    "result_kind": "row",
    "sql": "NOT regexp_matches(s_regex_not,'[0-9]')",
    "expected": "PASS"
  },
  {
    "id": "UNIQ_DISTINCT_COUNT_PROP_EXACT",
    "dimension": "Uniqueness",
    "business_label": "Proportion of distinct values is within range",
    "columns": [
      "id_prop"
    ],
    "result_kind": "aggregate",
    "sql": "(count(DISTINCT id_prop) * 1.0 / count(*)) BETWEEN 0.99 AND 1.0",
    "expected": "PASS"
  },
  {
    "id": "UNIQ_DISTINCT_COUNT_EXACT",
    "dimension": "Uniqueness",
    "business_label": "Number of distinct values is as expected",
    "columns": [
      "cat_distinct"
    ],
    "result_kind": "aggregate",
    "sql": "count(DISTINCT cat_distinct) BETWEEN 1 AND 12",
    "expected": "FAIL"
  },
  {
    "id": "MULT_FIELD_COMPARE_GT",
    "dimension": "Consistency",
    "business_label": "High value is greater than low value",
    "columns": [
      "n_gt_hi",
      "n_gt_lo"
    ],
    "result_kind": "row",
    "sql": "n_gt_hi > n_gt_lo",
    "expected": "FAIL"
  },
  {
    "id": "MULT_FIELD_COMPARE_EQ",
    "dimension": "Consistency",
    "business_label": "Two dates are equal",
    "columns": [
      "d_eq_a",
      "d_eq_b"
    ],
    "result_kind": "row",
    "sql": "d_eq_a = d_eq_b",
    "expected": "PASS"
  },
  {
    "id": "MULT_FIELD_DUPLICATES_EXACT",
    "dimension": "Uniqueness",
    "business_label": "No duplicate rows on the composite key",
    "columns": [
      "dup_a",
      "dup_b"
    ],
    "result_kind": "aggregate",
    "sql": "count(*) = count(DISTINCT (dup_a, dup_b))",
    "expected": "PASS"
  }
]
json.dump({'dataset': 'dataset', 'parquet_path': PARQUET_PATH, 'rules': RULES},
          open(RULES_PATH, 'w'), indent=2)
print('wrote', PARQUET_PATH)
print('wrote', RULES_PATH, '(%d rules: %d expected-fail, %d expected-pass)' %
      (len(RULES), sum(r['expected']=='FAIL' for r in RULES), sum(r['expected']=='PASS' for r in RULES)))

wrote C:\Users\Shadab\Personal_Projects\DUCKDB\dq-engine-compare\dataset.parquet
wrote C:\Users\Shadab\Personal_Projects\DUCKDB\dq-engine-compare\rules.json (20 rules: 10 expected-fail, 10 expected-pass)


In [3]:
# --- head, schema, row count, column count ---
from IPython.display import display
info = con.execute("PRAGMA table_info('dataset')").fetchall()
n_rows = con.execute('SELECT count(*) FROM dataset').fetchone()[0]
print('row count   :', f'{n_rows:,}')
print('column count:', len(info))
print('\nschema:')
for r in info: print(f'  {r[1]:20} {r[2]}')
print('\nhead:')
display(con.execute('SELECT * FROM dataset LIMIT 5').fetchdf())

row count   : 50,000,000
column count: 22

schema:
  c_any_nullable       VARCHAR
  c_any_required       VARCHAR
  n_max                DOUBLE
  n_min                DOUBLE
  n_sum                DOUBLE
  n_median             DOUBLE
  n_stddev             DOUBLE
  cat_mode             VARCHAR
  n_between            DOUBLE
  s_len                VARCHAR
  cat_in               VARCHAR
  cat_not_in           VARCHAR
  s_regex              VARCHAR
  s_regex_not          VARCHAR
  id_prop              BIGINT
  cat_distinct         BIGINT
  n_gt_hi              DOUBLE
  n_gt_lo              DOUBLE
  d_eq_a               DATE
  d_eq_b               DATE
  dup_a                BIGINT
  dup_b                BIGINT

head:


,c_any_nullable,c_any_required,n_max,n_min,n_sum,n_median,n_stddev,cat_mode,n_between,s_len,...,s_regex,s_regex_not,id_prop,cat_distinct,n_gt_hi,n_gt_lo,d_eq_a,d_eq_b,dup_a,dup_b
0,v0,None,441.77,182.56,7.935900e+02,568.82,400.60,A,500.0,xxxxxxxxxxxxxxxxxxxx,...,bad,charlie,0,999,0.0,708.66,2024-01-01,2024-01-01,0,0
1,v1,None,999999.00,320.04,4.290000e+00,744.68,462.94,B,500.0,xxxxxxxxxxxxxxxxxxxx,...,bad,bravo,1,999,0.0,290.33,2024-01-02,2024-01-02,1,2
2,v2,None,329.70,488.51,1.000000e+14,602.55,829.84,C,500.0,xxxxxxxxxxxxxxxxxxxx,...,bad,bravo,2,999,0.0,197.80,2024-01-03,2024-01-03,2,4
3,v3,None,487.14,531.87,4.387100e+02,168.58,377.64,B,500.0,xxxxxxxxxxxxxxxxxxxx,...,bad,alpha,3,999,0.0,284.01,2024-01-04,2024-01-04,3,6
4,v4,None,941.44,105.37,2.398700e+02,82.53,41.82,A,500.0,xxxxxxxxxxxxxxxxxxxx,...,bad,charlie,4,999,0.0,844.38,2024-01-05,2024-01-05,4,8
